# ITASORL — full end-to-end (Colab or local Jupyter)

Runs **pytest + all experiments** via `scripts/run_e2e.py`, records everything under
`results/runs/<timestamp>/`, and saves **`bundle.zip`** for offline review.

**Open on Google Colab (GPU):**
[colab_gpu.ipynb on GitHub](https://colab.research.google.com/github/iLevyTate/ITASORL/blob/expB2-survival-coupling/notebooks/colab_gpu.ipynb)

**Before you start (Colab):** Runtime → Change runtime type → **GPU**.

**Local Jupyter / VS Code:** open this file from the repo, set `MOUNT_DRIVE = False`, run all cells from repo root context (clone cell sets `REPO_DIR`).

**After the run, read:** `results/runs/<latest>/SUMMARY.md` (plain-English verdict).

In [ ]:
# --- Config ---
from pathlib import Path

REPO_URL = "https://github.com/iLevyTate/ITASORL.git"
BRANCH = "expB2-survival-coupling"  # or "main"

# Colab clones to /content; local Jupyter: point at your checkout
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
    REPO_DIR = "/content/ITASORL"
except ImportError:
    IN_COLAB = False
    REPO_DIR = str(Path.cwd().resolve())
    # If you opened the notebook from notebooks/, go up one level
    if not (Path(REPO_DIR) / "scripts" / "run_e2e.py").is_file():
        REPO_DIR = str(Path(REPO_DIR).parent)

RUN_MODE = "quick"          # "quick" (~30-60 min) | "full" (hours)
SKIP_STEPS = []             # e.g. ["expB2"] to skip longest step

MOUNT_DRIVE = IN_COLAB      # Colab: save to Drive; local: leave False
DRIVE_RESULTS = "/content/drive/MyDrive/ITASORL_results"
DOWNLOAD_BUNDLE = IN_COLAB  # browser download only works on Colab

In [ ]:
import os, subprocess, sys, shutil, time
from pathlib import Path

def sh(cmd, check=True):
    print(f"$ {cmd}", flush=True)
    subprocess.run(cmd, shell=True, check=check)

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_RESULTS, exist_ok=True)
elif not IN_COLAB:
    print(f"Local mode — repo: {REPO_DIR}")

In [ ]:
if IN_COLAB:
    if os.path.isdir(REPO_DIR):
        sh(f"cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}")
    else:
        sh(f"git clone --branch {BRANCH} --single-branch {REPO_URL} {REPO_DIR}")
else:
    print("Local mode — using existing checkout (skip clone)")
    if not (Path(REPO_DIR) / "itasorl").is_dir():
        raise FileNotFoundError(f"Expected itasorl/ under {REPO_DIR}; set REPO_DIR in config cell")

os.chdir(REPO_DIR)
sh("git log -1 --oneline")
print("REPO_DIR:", REPO_DIR)

In [ ]:
sh(f"{sys.executable} -m pip install -q --upgrade pip")
dev = os.path.join(REPO_DIR, "requirements-dev.txt")
if os.path.isfile(dev):
    sh(f"{sys.executable} -m pip install -q -r requirements-dev.txt")
else:
    sh(f"{sys.executable} -m pip install -q -r requirements.txt pytest")

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
cmd = [sys.executable, "scripts/run_e2e.py"]
if RUN_MODE == "quick":
    cmd.append("--quick")
elif RUN_MODE != "full":
    raise ValueError("RUN_MODE must be 'quick' or 'full'")
for step in SKIP_STEPS:
    cmd.extend(["--skip", step])

t0 = time.perf_counter()
subprocess.run(cmd, cwd=REPO_DIR, check=True)
print(f"Wall time: {(time.perf_counter() - t0) / 60:.1f} min")

In [ ]:
latest_ptr = Path(REPO_DIR) / "results" / "LATEST_RUN.txt"
RUN_DIR = Path(latest_ptr.read_text().strip())
BUNDLE = RUN_DIR / "bundle.zip"
SUMMARY = RUN_DIR / "SUMMARY.md"

print("Run directory:", RUN_DIR)
print("Bundle:", BUNDLE, "exists:", BUNDLE.is_file())
print("\n" + "=" * 72)
print(SUMMARY.read_text())
print("=" * 72)

In [ ]:
if MOUNT_DRIVE and BUNDLE.is_file():
    dest = Path(DRIVE_RESULTS) / f"{RUN_DIR.name}_bundle.zip"
    shutil.copy2(BUNDLE, dest)
    print("Saved to Drive:", dest)
    shutil.copy2(SUMMARY, Path(DRIVE_RESULTS) / f"{RUN_DIR.name}_SUMMARY.md")

if DOWNLOAD_BUNDLE and BUNDLE.is_file():
    from google.colab import files
    files.download(str(BUNDLE))
    print("Browser download started for bundle.zip")
elif BUNDLE.is_file():
    print("Local bundle path:", BUNDLE)
    print("Open SUMMARY:", SUMMARY)

## What's in the bundle

| File | Purpose |
|------|---------|
| `SUMMARY.md` | Plain-English outcome — **read this first** |
| `manifest.json` | Step timings, status, artifact index |
| `combined.log` | Full stdout from every step |
| `steps/*.json` | Parsed metrics (AUROC per drift, etc.) |
| `artifacts/` | Figures + `expB2_results.json` |

Unzip locally and open `SUMMARY.md` to decide whether the organism encoded world identity.